# Lecture 2 - Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules (these will be checked in the model answer review next week)
1. Every bar chart **must have a zero baseline** - no exceptions (SWD p.51)
2. Every chart **must have an insight title**, not a topic title (SWD p.29)
3. Aim for **professional quality** - clean background, readable font, no clutter
4. Horizontal bars for long category names (SWD p.57)

---


## Setup - Run this cell first


In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
               'Life_Expectancy','Freedom','Generosity','Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())
df.columns

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


Index(['Country', 'Region', 'Happiness_Score', 'GDP', 'Social_Support',
       'Life_Expectancy', 'Freedom', 'Generosity', 'Corruption'],
      dtype='object')

In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print('Regions in dataset:')
print(df['Region'].value_counts())
print('\nScore range:', df['Happiness_Score'].min(), '-', df['Happiness_Score'].max())
print('\nBottom 10 countries:')
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 - 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 - Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score', ascending=False))

print(region_avg)

# Step 2: Build your chart
top_region = region_avg.iloc[0]['Region']
top_score = region_avg.iloc[0]['Happiness_Score']
region_colors = ['#1f4e79' if region == top_region else '#9bb7d4' for region in region_avg['Region']]
bar_labels = [f'{score:.2f}' if region != top_region else ''
              for region, score in zip(region_avg['Region'], region_avg['Happiness_Score'])]

fig1 = go.Figure(
    go.Bar(
        x=region_avg['Happiness_Score'],
        y=region_avg['Region'],
        orientation='h',
        marker_color=region_colors,
        text=bar_labels,
        textposition='outside'
    )
)

fig1.update_layout(
    title='Western Europe leads regional happiness, showing a clear advantage over every other region',
    xaxis_title='Average Happiness Score',
    yaxis_title='',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12),
    margin=dict(l=20, r=95, t=80, b=20)
)

fig1.update_xaxes(range=[0, region_avg['Happiness_Score'].max() + 1.4], gridcolor='#d9d9d9', zeroline=True)
fig1.update_yaxes(autorange='reversed')
fig1.add_annotation(
    x=top_score,
    y=top_region,
    text=f'Highest regional average: {top_score:.2f}',
    showarrow=True,
    arrowhead=2,
    ax=80,
    ay=0
)

fig1.show()


                         Region  Happiness_Score
8                Western Europe         7.085533
4         North America and ANZ         7.018250
0    Central and Eastern Europe         6.338143
1                     East Asia         5.966000
2   Latin America and Caribbean         5.699000
6                Southeast Asia         5.695250
3  Middle East and North Africa         4.943333
7            Sub-Saharan Africa         4.064714
5                    South Asia         3.618250


---
## Task 2 - Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score').reset_index(drop=True)
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
fig2 = px.bar(
    combined,
    x='Country',
    y='Happiness_Score',
    color='Group',
    color_discrete_map={'Bottom 8': '#c44e52', 'Top 8': '#4c72b0'},
    text='Happiness_Score'
)

fig2.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig2.update_layout(
    title='The happiest countries score roughly twice as high as the least happy countries',
    xaxis_title='',
    yaxis_title='Happiness Score',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12),
    legend_title='',
    margin=dict(l=20, r=20, t=80, b=80)
)

fig2.update_yaxes(range=[0, combined['Happiness_Score'].max() + 1], gridcolor='#d9d9d9', zeroline=True)
fig2.update_xaxes(tickangle=-40)
fig2.add_shape(
    type='line',
    x0=7.5,
    x1=7.5,
    y0=0,
    y1=combined['Happiness_Score'].max() + 0.5,
    line=dict(color='#444444', width=2, dash='dash')
)
fig2.add_annotation(
    x=7.5,
    y=combined['Happiness_Score'].max() + 0.55,
    text='Gap between bottom 8 and top 8',
    showarrow=False
)
fig2.add_hline(
    y=global_avg,
    line_dash='dot',
    line_color='#666666',
    annotation_text=f'Global average: {global_avg:.2f}',
    annotation_position='top left'
)

fig2.show()


Global average: 5.81


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [7]:
# Stretch goal - grouped bar chart
# --------------------------------

# Step 1: Keep only the required regions and compute average sub-factor scores
selected_regions = ['Western Europe', 'Latin America', 'East Asia',
                    'Sub-Saharan Africa', 'South Asia']

factor_avg = (df[df['Region'].isin(selected_regions)]
              .groupby('Region')[['GDP', 'Freedom']]
              .mean()
              .reindex(selected_regions)
              .reset_index())

print(factor_avg)

# Step 2: Convert to long format for a grouped bar chart
factor_long = factor_avg.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Sub-factor',
    value_name='Average score'
)

factor_long['Sub-factor'] = factor_long['Sub-factor'].replace({
    'GDP': 'GDP contribution',
    'Freedom': 'Freedom contribution'
})

# Step 3: Build the chart
fig3 = px.bar(
    factor_long,
    x='Region',
    y='Average score',
    color='Sub-factor',
    barmode='group',
    text='Average score',
    color_discrete_map={
        'GDP contribution': '#2f5d7c',
        'Freedom contribution': '#e07a5f'
    }
)

fig3.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig3.update_layout(
    title='Western Europe pairs the strongest GDP contribution with high freedom scores',
    xaxis_title='',
    yaxis_title='Average contribution to happiness score',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12),
    legend_title='',
    margin=dict(l=20, r=20, t=80, b=90)
)

fig3.update_yaxes(range=[0, factor_long['Average score'].max() + 0.25], gridcolor='#d9d9d9', zeroline=True)
fig3.update_xaxes(tickangle=-25)

fig3.show()


               Region        GDP   Freedom
0      Western Europe  10.963533  0.896067
1       Latin America        NaN       NaN
2           East Asia  10.312000  0.754000
3  Sub-Saharan Africa   7.082000  0.599429
4          South Asia   7.152250  0.546250
